# Early earthquake magnitude: high-tail experiment record

**Objective:** improve high-Mw magnitude error at 1, 3 and 5 seconds while preserving bulk MAE, median error and worst-error performance.

**Current conclusion:** no novel winning method is established. The strongest existing practical baseline adds instrument/site information; it improves ordinary and M≥4 errors on reused INSTANCE validation. That is not evidence for rare Mw≥6/7 generalization. GeoNet waveforms are being qualified to address this gap.

This notebook reads only committed public aggregates. It performs no downloads, decoding, training or AWS actions. Actual waveforms, catalogue identifiers, requests and resource credentials remain private.


## Reproduce the completed acquisition checks

Run from the repository root or this notebook’s directory. The loader finds only the published result directory and reads two fixed aggregate files. Both sources were reviewed and committed before their sole actual invocations. All 65 original TRAIN/DEV events remain in the denominator; 31 TEST events were excluded.


In [1]:
from pathlib import Path
import json

required = ("geonet_POST_transport_outcome_20261012.json", "geonet_POST_header_outcome_20261012.json")
result_dir = next((base / rel for base in (Path.cwd(), *Path.cwd().parents)
                   for rel in ("results/2026-10-09", "work/repo/results/2026-10-09")
                   if all((base / rel / name).is_file() for name in required)), None)
if result_dir is None:
    raise FileNotFoundError("Open this notebook inside the published Earthquake repository.")
transport = json.loads((result_dir / required[0]).read_text())
headers = json.loads((result_dir / required[1]).read_text())
assert transport["actual_invocations"] == headers["actual_invocations"] == 1
assert transport["audit"]["valid"] and transport["audit"]["whole_complete"]
assert transport["audit"]["qualified_batches"] == headers["supported_batches"] == 17
assert headers["retained_bytes"] == transport["audit"]["aggregate"]["retained_response_bytes"]
print(json.dumps({"fixed_events": 65, "TRAIN": 41, "DEV": 24, "TEST_excluded": 31,
                  "complete_requests": 17, "downloaded_bytes": headers["retained_bytes"],
                  "structurally_supported_records": headers["supported_records"],
                  "advertised_samples": headers["advertised_samples"]}, indent=2))


{
  "fixed_events": 65,
  "TRAIN": 41,
  "DEV": 24,
  "TEST_excluded": 31,
  "complete_requests": 17,
  "downloaded_bytes": 539648,
  "structurally_supported_records": 1054,
  "advertised_samples": 227658
}


## Separate transport success from scientific evidence

HTTP200 and a valid record layout do not establish continuous three-component prefixes, physical units, online arrival time, independent earthquake sequences or an accuracy gain. The next record-by-record decoder retains each gap, unknown and metadata-family ambiguity; it excludes every sample at or after its 1/3/5-second cutoff. The Linux decoder uses a separate pinned environment and task-owned memory/time limits.


In [2]:
assert transport["decoded_or_usable_rows_verified"] == 0
assert not headers["payload_decode_or_training_performed"]
assert not headers["waveform_coverage_verified"]
assert not headers["novelty_verified"]
print("At this recorded milestone: actual prefix qualification and training are still pending.")


At this recorded milestone: actual prefix qualification and training are still pending.


## What the next experiments must establish

1. Qualify the fixed waveform cohort at all three horizons without repairing failed cases by changing station or family.
2. Run a small waveform-plus-metadata model against matched metadata-only and event-label-permutation controls, with two seeds. This checks information content, not novelty.
3. Test a prospectively specified receiver-activity persistence likelihood against full-prefix inference, ordinary activity features and the earlier peak-growth likelihood. Persistence is an observable receiver state; it is not a measurement of physical rupture completion. Closest duration/survival prior art must be attributed.
4. Use a larger, separately sealed, sequence-aware evaluation for simultaneous high-Mw improvement and bulk preservation. The stratified 24-event DEV sample is insufficient for that claim. Published comparisons need matched data, magnitude scale, clock and preprocessing.

Every negative result stays in the record. A statistically inconclusive or bulk-worsening candidate is rejected, even if its tail point estimate looks better.


## Retained practical baseline

Earlier two-seed full-TRAIN INSTANCE results (reused validation; 13 M≥4 events, only one M≥5):

| Horizon | Bulk MAE: baseline → instrument/site | M≥4 MAE: baseline → instrument/site |
|---|---:|---:|
| 1 s | 0.395628 → 0.365343 | 0.852221 → 0.686116 |
| 3 s | 0.346881 → 0.315881 | 0.719547 → 0.551167 |
| 5 s | 0.317999 → 0.286190 | 0.633145 → 0.486910 |

These are useful controls, not a new probability principle or a matched published-benchmark win. See the repository’s `earthquake_instrument_full_results.md`, `earthquake_method_status.md` and retained negative experiment reports for details.
